### What is the error handling policy in Python

Python uses a different approach to error checking than many other common languages. Instead of trying to beforehand check that all the inputs are of correct type and then contents of input variables are sensible for some operations, Python first tries the operations and then checks whether they caused any exceptions. This is partly what duck typing is about: a function works for a set of inputs if all the operations in the function body make sense for those inputs. So, that’s why the parameters of functions aren’t specified to be of any certain type.

## "Try First, Check Later" — Simply

This describes Python's overall philosophy for error checking, and connects directly back to duck typing — which you actually already met, briefly, in your `sum()` deep-dive earlier!

---

### Two Different Philosophies, Contrasted

**Approach 1 — Check FIRST, then act (many other languages)**

Before doing anything, verify **everything** is correct:

```
Is this the right type?      → check
Is this value in range?        → check
Is this not null/empty?          → check
... (all checks pass) ...
NOW actually perform the operation
```

**Approach 2 — Try FIRST, deal with problems if they occur (Python)**

Just attempt the operation directly. If something's wrong, an exception will naturally occur — **catch that**, rather than pre-checking:

```
Just DO the operation
   ↓
Did it work?  → great, continue
Did it raise an exception?  → handle it THEN
```

This second style has an actual name in the Python community: **EAFP** — *"Easier to Ask Forgiveness than Permission."* (The opposite style, checking first, is sometimes called **LBYL** — *"Look Before You Leap."*)

---

### A Concrete Example — Both Styles Side by Side

**Check-first style ("Look Before You Leap"):**

```python
def get_value(d, key):
    if key in d:                 # CHECK first
        return d[key]
    else:
        return None
```

**Try-first style ("Easier to Ask Forgiveness"):**

```python
def get_value(d, key):
    try:
        return d[key]            # JUST TRY it
    except KeyError:
        return None
```

Both give the **same result** — but the second one matches how Python's standard library itself tends to behave, and how the language encourages you to write code.

---

### Connecting This to Duck Typing — Where You Met It Before

Remember this exact phrase from your `sum()` internals discussion:

> *"if it walks like a duck and quacks like a duck, treat it as a duck"* — `sum()` doesn't check `type(thing) == generator`. It just tries `iter(thing)` — and if that succeeds, it proceeds.

**This is EXACTLY the mechanism being described here, generalized.** `sum()` doesn't pre-verify *"is this specifically a list, or a generator, or a tuple?"* — it just **tries** calling `iter()` on it. If that works, great — proceed. If it doesn't, an exception naturally happens, and *that's* the error signal.

```python
sum([1, 2, 3])              # tries it → iter() works → proceeds
sum(x for x in range(3))     # tries it → iter() works → proceeds
sum(5)                       # tries it → iter() FAILS → TypeError naturally raised
```

Python never asked "what TYPE are you?" beforehand — it just **attempted the operation**, and let the attempt itself reveal whether it was valid.

---

### "A Function Works for a Set of Inputs IF All the Operations Make Sense"

This is the precise duck-typing definition. A function is considered "compatible" with **whatever input** happens to support the operations the function actually performs — **not** based on some declared, checked type.

```python
def double_everything(items):
    return [x * 2 for x in items]      # only REQUIRES: iterable, and * works on elements
```

```python
double_everything([1, 2, 3])          # ✓ list — works
double_everything((1, 2, 3))            # ✓ tuple — works
double_everything("abc")                  # ✓ string! "a"*2 = "aa" — even THIS works!
double_everything({1, 2, 3})               # ✓ set — works
```

`double_everything` never asked *"are you specifically a list?"* — it just **assumed** you could be iterated and multiplied, and **tried**. Any type that happens to genuinely support those operations sails through — string included, which the function's author may not have even anticipated!

```python
double_everything(5)     # ✗ TypeError: 'int' object is not iterable
```

Only when an incompatible type is actually **used**, does the mismatch surface — as a natural exception, exactly where the operation fails, not as a pre-emptive type check at the function's entrance.

---

### "That's Why Function Parameters Aren't Specified to Be of Any Certain Type"

This explains something you've probably noticed all along — your own `detect_ranges(L)`, `summary(filename)`, `area_of_triangle(base, height)` **never declared** what type their parameters must be:

```python
def area_of_triangle(base, height):     # no ": int" or ": float" anywhere!
    return 0.5 * base * height
```

Compare with a statically-typed language (Java-style thinking):

```java
double areaOfTriangle(double base, double height) {   // TYPES REQUIRED
    return 0.5 * base * height;
}
```

Python **doesn't require this**, precisely because of the try-first philosophy: `area_of_triangle` doesn't need to **declare** "I require floats" — it just needs `*` to make sense on whatever gets passed in. If someone passes strings, `*` will simply behave differently (or fail), and **that's** the natural feedback — no upfront type declaration was ever needed to make this work correctly for legitimate inputs.

---

### Why This Design Is a Deliberate Choice, Not Laziness

**Benefit 1 — More flexible code, for free:**
```python
area_of_triangle(3, 4)              # works — ints
area_of_triangle(3.5, 4.2)            # works — floats
area_of_triangle(Fraction(1,2), 4)      # works — even Fractions, from your earlier lesson!
```

The function never anticipated `Fraction` specifically — but since `*` and `0.5 *` make sense for it, it **just works**, with zero extra code.

**Benefit 2 — Matches the whole "duck typing" spirit of `zip()`, `sum()`, `for` loops** you've been learning throughout this conversation: Python consistently prefers *"does the operation work?"* over *"is this the officially blessed type?"*

---

### The Trade-off — Connecting to Your Java Discussion

This is the **exact same trade-off** from your last question, just viewed from a different angle:

| | Check-first (Java-style) | Try-first (Python-style) |
|---|---|---|
| Errors caught | early, often before running | only when the actual operation runs |
| Flexibility | rigid — must match declared types | flexible — works for ANY compatible type |
| Safety net | strong, automatic (compiler) | relies on YOU writing good `try`/`except` |

Same tension as before: Python trades upfront safety for flexibility and simplicity — which is exactly why your earlier concern about `except Exception:` hiding bugs matters so much. Since Python **never** checks types upfront, your exception handling really is the only thing standing between "the try-first philosophy working beautifully" and "a bug slipping through unnoticed."

---

### The One-Sentence Summary

> Instead of verifying types and values **before** running an operation, Python just **attempts** the operation directly and reacts to whatever exception naturally occurs if it fails — this is **duck typing**: a function works for **any** input where its internal operations happen to make sense, regardless of the input's declared type, which is exactly why Python function parameters never require type declarations, and exactly why `sum()`, `zip()`, and your own functions work seamlessly across lists, tuples, strings, and custom objects alike. 🎯